# Step 22: MandiMitra ML V3 — Price Direction & Cross-Mandi Intelligence

This experiment investigates whether machine learning can deliver more reliable decision support by predicting **Price Direction** (INCREASE, STABLE, DECREASE) over the next ~3 market observations rather than competing with persistence on exact point forecasts.

### Key Principles
- **Empirically Justified Movement Thresholds**: Set based on training set volatility (Rice: ±1.5%, Tomato: ±5.0%, Wheat: ±1.5%, Cotton: ±1.0%).
- **Cross-Mandi Information**: Incorporates crop-wide price stats, market rank, spread, and momentum across active APMCs.
- **Foundation Model Integration**: Tests Chronos-2 forecast signals as an input feature.
- **Strict Causality**: Zero data leakage; only historical observations $\le T$ are used.
- **Decision Relevance**: Evaluates precision of 'WAIT' (Increase) and 'SELL' (Decrease/Stable) signals.

## 1. Run the Complete Direction Classification Benchmark

In [1]:
import sys
from pathlib import Path
import pandas as pd
BASE_DIR = Path('..').resolve()
if str(BASE_DIR) not in sys.path: sys.path.insert(0, str(BASE_DIR))

from src.direction_evaluation import run_direction_experiment
df_comp, df_abl = run_direction_experiment()


MANDIMITRA ML V3: PRICE DIRECTION EXPERIMENT
Saved outputs/final/direction_threshold_analysis.csv

Evaluating crop: RICE...
Train size: 1342 | Test size: 327
Test class distribution: Decrease: 41, Stable: 233, Increase: 53



Evaluating crop: TOMATO...
Train size: 1691 | Test size: 411
Test class distribution: Decrease: 148, Stable: 156, Increase: 107



Evaluating crop: WHEAT...
Train size: 9712 | Test size: 2367
Test class distribution: Decrease: 502, Stable: 1360, Increase: 505



Evaluating crop: COTTON...
Train size: 326 | Test size: 80
Test class distribution: Decrease: 19, Stable: 42, Increase: 19



Saved all 6 outputs to outputs/final/

BEST DIRECTION MODEL PER CROP SUMMARY
Crop: Rice     | Best Model: gradient_boosting      | Macro F1: 0.4080 | Balanced Acc: 0.5176
Crop: Tomato   | Best Model: random_forest          | Macro F1: 0.4931 | Balanced Acc: 0.5260
Crop: Wheat    | Best Model: random_forest          | Macro F1: 0.5717 | Balanced Acc: 0.5937
Crop: Cotton   | Best Model: gradient_boosting      | Macro F1: 0.5517 | Balanced Acc: 0.5827


## 2. Empirical Threshold Analysis

In [2]:
th_df = pd.read_csv(BASE_DIR / 'outputs' / 'final' / 'direction_threshold_analysis.csv')
display(th_df)


,crop,n_train_samples,mean_abs_pct_change,median_abs_pct_change,p25_abs_pct_change,p75_abs_pct_change,exact_zero_change_pct,selected_threshold_pct,train_increase_pct,train_stable_pct,train_decrease_pct,justification
0,Rice,1342,2.95,0.00,0.00,3.37,68.41,1.5,14.90,70.72,14.38,Selected ±1.5% to separate true trends from 68...
1,Tomato,1691,19.39,14.81,0.61,27.27,23.89,5.0,35.42,31.05,33.53,Selected ±5.0% to separate true trends from 23...
2,Wheat,9712,2.97,1.65,0.04,4.00,24.90,1.5,26.27,47.73,26.00,Selected ±1.5% to separate true trends from 24...
3,Cotton,326,1.16,0.39,0.00,1.41,38.96,1.0,19.02,66.87,14.11,Selected ±1.0% to separate true trends from 39...


## 3. Model Comparison Across Crops (Baselines vs. ML Classifiers)

In [3]:
comp_df = pd.read_csv(BASE_DIR / 'outputs' / 'final' / 'direction_model_comparison.csv')
display(comp_df.sort_values(by=['crop', 'f1_macro'], ascending=[True, False]))


,crop,model,model_type,accuracy,balanced_accuracy,f1_macro,f1_weighted,precision_macro,recall_macro,precision_increase,recall_increase,f1_increase,precision_stable,recall_stable,f1_stable,precision_decrease,recall_decrease,f1_decrease,wait_precision,sell_precision
26,Cotton,gradient_boosting,ML Classifier,0.6000,0.5827,0.5517,0.5847,0.6257,0.5827,0.7143,0.2632,0.3846,0.6923,0.6429,0.6667,0.4706,0.8421,0.6038,0.7143,0.8082
25,Cotton,random_forest,ML Classifier,0.4375,0.4891,0.4349,0.4285,0.4503,0.4891,0.3200,0.4211,0.3636,0.6190,0.3095,0.4127,0.4118,0.7368,0.5283,0.3200,0.8000
24,Cotton,logistic_regression,ML Classifier,0.4000,0.4942,0.3937,0.3565,0.4533,0.4942,0.3600,0.4737,0.4091,0.6364,0.1667,0.2642,0.3636,0.8421,0.5079,0.3600,0.8182
27,Cotton,hist_gradient_boosting,ML Classifier,0.5125,0.3830,0.3691,0.4510,0.4473,0.3830,0.4286,0.1579,0.2308,0.5385,0.8333,0.6542,0.3750,0.1579,0.2222,0.4286,0.7808
23,Cotton,momentum,Baseline,0.4375,0.2970,0.2617,0.3725,0.2507,0.2970,0.0909,0.0526,0.0667,0.5500,0.7857,0.6471,0.1111,0.0526,0.0714,0.0909,0.7391
21,Cotton,majority_class,Baseline,0.5250,0.3333,0.2295,0.3615,0.1750,0.3333,0.0000,0.0000,0.0000,0.5250,1.0000,0.6885,0.0000,0.0000,0.0000,0.0000,0.7625
22,Cotton,persistence,Baseline,0.5250,0.3333,0.2295,0.3615,0.1750,0.3333,0.0000,0.0000,0.0000,0.5250,1.0000,0.6885,0.0000,0.0000,0.0000,0.0000,0.7625
2,Rice,momentum,Baseline,0.7431,0.5289,0.5212,0.7585,0.5173,0.5289,0.3051,0.3396,0.3214,0.9724,0.9056,0.9378,0.2745,0.3415,0.3043,0.3051,0.8694
5,Rice,gradient_boosting,ML Classifier,0.4312,0.5176,0.4080,0.4803,0.4715,0.5176,0.1069,0.2642,0.1522,0.9375,0.3863,0.5471,0.3700,0.9024,0.5248,0.1069,0.8010
4,Rice,random_forest,ML Classifier,0.3303,0.4619,0.3640,0.3709,0.4736,0.4619,0.0824,0.2830,0.1277,0.9062,0.2489,0.3906,0.4321,0.8537,0.5738,0.0824,0.7379


## 4. Feature Ablation Study (Historical vs Cross-Mandi vs Chronos-2)

In [4]:
abl_df = pd.read_csv(BASE_DIR / 'outputs' / 'final' / 'direction_ablation_results.csv')
display(abl_df)


,crop,model,feature_group,n_features,accuracy,balanced_accuracy,f1_macro,precision_macro,recall_macro,wait_precision,sell_precision
0,Rice,gradient_boosting,historical,37,0.3425,0.4518,0.3044,0.4134,0.4518,0.1071,0.8189
1,Rice,gradient_boosting,cross_mandi,51,0.3180,0.4252,0.2866,0.4156,0.4252,0.1695,0.8396
2,Rice,gradient_boosting,chronos,43,0.3242,0.4433,0.3000,0.4126,0.4433,0.0804,0.7953
3,Rice,gradient_boosting,all,57,0.4312,0.5176,0.4080,0.4715,0.5176,0.1069,0.8010
4,Tomato,random_forest,historical,37,0.5085,0.5312,0.4986,0.5271,0.5312,0.4270,0.8670
5,Tomato,random_forest,cross_mandi,51,0.5134,0.5382,0.5039,0.5434,0.5382,0.4225,0.8750
6,Tomato,random_forest,chronos,43,0.4915,0.5142,0.4798,0.5068,0.5142,0.4205,0.8596
7,Tomato,random_forest,all,57,0.5036,0.5260,0.4931,0.5279,0.5260,0.4167,0.8615
8,Wheat,random_forest,historical,37,0.6033,0.5973,0.5745,0.5661,0.5973,0.4414,0.8760
9,Wheat,random_forest,cross_mandi,51,0.6012,0.5964,0.5732,0.5663,0.5964,0.4296,0.8780


## 5. Market-Level Generalization Performance

In [5]:
mkt_df = pd.read_csv(BASE_DIR / 'outputs' / 'final' / 'direction_market_performance.csv')
display(mkt_df.head(20))


,crop,market,n_samples,best_model,accuracy,actual_decrease_count,actual_stable_count,actual_increase_count,predicted_decrease_count,predicted_stable_count,predicted_increase_count
0,Rice,APMC Alibagh,111,gradient_boosting,0.4054,0,108,3,7,48,56
1,Rice,APMC Murud,111,gradient_boosting,0.4054,0,108,3,7,48,56
2,Rice,APMC Palghar,105,gradient_boosting,0.4857,41,17,47,86,0,19
3,Tomato,APMC Kamthi,104,random_forest,0.5577,31,51,22,37,34,33
4,Tomato,APMC Panvel,106,random_forest,0.5000,38,41,27,62,2,42
5,Tomato,Pune(Manjri),107,random_forest,0.5047,41,33,33,41,6,60
6,Tomato,Pune(Pimpri),94,random_forest,0.4468,38,31,25,21,28,45
7,Wheat,APMC Akola,93,random_forest,0.5054,20,52,21,21,43,29
8,Wheat,APMC Amarawati,86,random_forest,0.6512,12,63,11,17,47,22
9,Wheat,APMC Chattrapati Sambhajinagar,90,random_forest,0.4444,27,33,30,33,19,38


## 6. Confusion Matrices

In [6]:
cm_df = pd.read_csv(BASE_DIR / 'outputs' / 'final' / 'direction_confusion_matrices.csv')
display(cm_df)


,crop,model,tn_dec_dec,dec_as_stb,dec_as_inc,stb_as_dec,stb_as_stb,stb_as_inc,inc_as_dec,inc_as_stb,tp_inc_inc
0,Rice,majority_class,0,41,0,0,233,0,0,53,0
1,Rice,persistence,0,41,0,0,233,0,0,53,0
2,Rice,momentum,14,0,27,8,211,14,29,6,18
3,Rice,logistic_regression,41,0,0,177,56,0,44,7,2
4,Rice,random_forest,35,0,6,14,58,161,32,6,15
5,Rice,gradient_boosting,37,0,4,30,90,113,33,6,14
6,Rice,hist_gradient_boosting,34,0,7,53,82,98,34,6,13
7,Tomato,majority_class,0,0,148,0,0,156,0,0,107
8,Tomato,persistence,0,148,0,0,156,0,0,107,0
9,Tomato,momentum,48,33,67,50,71,35,65,26,16


## 7. Direction Inference Interface Demo

In [7]:
from src.direction_inference import predict_price_direction
import json

for crop, mkt, pr in [
    ('rice', 'APMC Alibagh', 3500.0),
    ('tomato', 'APMC Kamthi', 2770.0),
    ('wheat', 'APMC Nagpur', 2650.0),
    ('cotton', 'APMC Hinganghat', 7900.0)
]:
    res = predict_price_direction(crop, mkt, pr, '2026-09-04')
    print(f'=== {crop.upper()} ===')
    print(json.dumps(res, indent=2))


=== RICE ===
{
  "crop": "rice",
  "market": "APMC Alibagh",
  "current_price": 3500.0,
  "predicted_direction": "STABLE",
  "confidence": 0.457,
  "probabilities": {
    "decrease": 0.194,
    "stable": 0.457,
    "increase": 0.349
  },
  "forecast_horizon": "3 market observations",
  "movement_threshold_pct": 1.5,
  "interpretation": "Price is likely to stable by more than 1.5% over ~3 observations with 45.7% confidence.",
  "experimental_method": "direction_v3_classifier"
}


=== TOMATO ===
{
  "crop": "tomato",
  "market": "APMC Kamthi",
  "current_price": 2770.0,
  "predicted_direction": "STABLE",
  "confidence": 0.451,
  "probabilities": {
    "decrease": 0.214,
    "stable": 0.451,
    "increase": 0.335
  },
  "forecast_horizon": "3 market observations",
  "movement_threshold_pct": 5.0,
  "interpretation": "Price is likely to stable by more than 5.0% over ~3 observations with 45.1% confidence.",
  "experimental_method": "direction_v3_classifier"
}


=== WHEAT ===
{
  "crop": "wheat",
  "market": "APMC Nagpur",
  "current_price": 2650.0,
  "predicted_direction": "STABLE",
  "confidence": 0.503,
  "probabilities": {
    "decrease": 0.252,
    "stable": 0.503,
    "increase": 0.245
  },
  "forecast_horizon": "3 market observations",
  "movement_threshold_pct": 1.5,
  "interpretation": "Price is likely to stable by more than 1.5% over ~3 observations with 50.3% confidence.",
  "experimental_method": "direction_v3_classifier"
}
=== COTTON ===
{
  "crop": "cotton",
  "market": "APMC Hinganghat",
  "current_price": 7900.0,
  "predicted_direction": "INCREASE",
  "confidence": 0.548,
  "probabilities": {
    "decrease": 0.124,
    "stable": 0.327,
    "increase": 0.548
  },
  "forecast_horizon": "3 market observations",
  "movement_threshold_pct": 1.0,
  "interpretation": "Price is likely to increase by more than 1.0% over ~3 observations with 54.8% confidence.",
  "experimental_method": "direction_v3_classifier"
}
